# Mini-projet : mini-moteur 3D fil de fer

| Module | Durée | Prérequis |
|---|---|---|
| M3 – Matrices et transformations | 150 min | M2 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le club informatique d'une médiathèque veut montrer aux adolescents « ce qu'il y a sous le capot » d'un jeu vidéo en 3D. Le défi : **afficher un objet en 3D, en fil de fer, et le faire tourner avec des curseurs, sans aucune bibliothèque 3D**. Seulement NumPy et Matplotlib.

Les questions de l'animateur :

1. Comment représenter un objet 3D dans le programme ?
2. Comment le faire tourner dans l'espace ?
3. Comment le dessiner sur un écran, qui est plat ?
4. Pourquoi les objets lointains paraissent-ils plus petits, et comment le reproduire ?

**Activité** : Listez les hypothèses simplificatrices que vous faites. Par exemple : l'objet est-il rigide ? La caméra bouge-t-elle ? Dessine-t-on les faces cachées ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Un cube d'arête 2 est centré à 4 unités devant la caméra. Sa face avant est donc à la distance 3, sa face arrière à la distance 5. Sur l'écran, la face avant paraît plus grande. Pariez sur le rapport de taille.

In [ ]:
pari_rapport_avant_arriere = 1.25   # taille apparente face avant / face arrière (modifiez)

## ③ Abstraire

### Représenter l'objet

Un objet en fil de fer, ce sont des **sommets** (des points de l'espace) et des **arêtes** (des paires de sommets à relier). On range les sommets en colonnes dans une matrice de taille $3 \times n$, comme au notebook 3.1.

### Tourner dans l'espace

Une rotation autour de l'un des trois axes laisse cet axe fixe et tourne le plan des deux autres comme en 2D :

$$
R_x(\alpha) = \begin{pmatrix} 1 & 0 & 0 \\ 0 & \cos\alpha & -\sin\alpha \\ 0 & \sin\alpha & \cos\alpha \end{pmatrix},
\quad
R_y(\beta) = \begin{pmatrix} \cos\beta & 0 & \sin\beta \\ 0 & 1 & 0 \\ -\sin\beta & 0 & \cos\beta \end{pmatrix},
\quad
R_z(\gamma) = \begin{pmatrix} \cos\gamma & -\sin\gamma & 0 \\ \sin\gamma & \cos\gamma & 0 \\ 0 & 0 & 1 \end{pmatrix}
$$

Comme au notebook 3.2, l'ordre compte : $R_x R_y \neq R_y R_x$ en général.

### Projeter sur l'écran

Deux choix, deux concepts :

- **Projection orthographique** : on oublie la profondeur $z$. C'est une application linéaire, de matrice $\begin{pmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \end{pmatrix}$. Simple, mais sans effet de profondeur.
- **Projection en perspective** : un point est vu à travers l'écran, sur la droite qui le relie à l'œil.

![Schéma : l'œil, l'écran à la distance f, et un point à la profondeur z ; par triangles semblables, x' = f x / z](../../assets/schemas/schema-perspective.svg)

Avec la caméra placée à la distance $d$ du centre de l'objet, un point $(x\,;\,y\,;\,z)$ s'affiche en

$$
x' = \frac{f\,x}{z + d}, \qquad y' = \frac{f\,y}{z + d}.
$$

**Attention :** la division par $z + d$ rend la perspective **non linéaire**. Les moteurs 3D retrouvent un calcul matriciel grâce aux coordonnées homogènes du notebook 3.3, en dimension 4, puis font une seule division à la fin.

**Activité** : Avec $f = 1$ et $d = 4$, où s'affichent les sommets $(1\,;\,1\,;\,-1)$ (face avant) et $(1\,;\,1\,;\,1)$ (face arrière) ? Déduisez-en le rapport de taille des deux faces.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### L'objet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

# Les 8 sommets du cube [-1, 1]^3, en colonnes : forme (3, 8)
cube = np.array(list(product([-1, 1], repeat=3)), dtype=float).T
# Deux sommets sont reliés s'ils diffèrent par une seule coordonnée : 12 arêtes
aretes_cube = [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(cube[:, i] != cube[:, j]) == 1]
print("Sommets :", cube.shape, "  Arêtes :", len(aretes_cube))

### Les rotations et les projections

In [ ]:
def Rx(a):
    a = np.radians(a); c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def Ry(b):
    b = np.radians(b); c, s = np.cos(b), np.sin(b)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def Rz(g):
    g = np.radians(g); c, s = np.cos(g), np.sin(g)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def tourner(sommets, ax_deg=0, ay_deg=0, az_deg=0):
    # Ordre choisi : d'abord autour de z, puis de y, puis de x
    return Rx(ax_deg) @ Ry(ay_deg) @ Rz(az_deg) @ sommets

def projection_ortho(sommets):
    return np.array([[1, 0, 0], [0, 1, 0]]) @ sommets

def projection_perspective(sommets, d=4.0, f=2.0):
    profondeur = sommets[2] + d
    return f * sommets[:2] / profondeur, profondeur

def tracer(ax, points_2d, aretes, titre="", visibles=None, couleur=None):
    couleur = couleur or LIOUAI["navy"]
    for k, (i, j) in enumerate(aretes):
        if visibles is not None and not (visibles[i] and visibles[j]):
            continue
        ax.plot(*points_2d[:, [i, j]], color=couleur, lw=1.8)
    ax.plot(*points_2d, "o", color=LIOUAI["vert"], ms=4)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
    ax.set_title(titre, fontsize=10)

### Orthographique ou perspective ?

In [ ]:
vue = tourner(cube, ax_deg=25, ay_deg=35)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
tracer(axes[0], projection_ortho(cube), aretes_cube, "Orthographique, sans rotation")
tracer(axes[1], projection_ortho(vue), aretes_cube, "Orthographique, après rotation")
p2d, _ = projection_perspective(vue, d=4, f=2)
tracer(axes[2], p2d, aretes_cube, "Perspective (d = 4), après rotation")
plt.tight_layout(); plt.show()

### Vérifier que l'objet reste rigide

Une rotation ne doit déformer ni étirer l'objet : toutes les arêtes doivent garder la longueur 2.

In [ ]:
def longueurs_aretes(sommets, aretes):
    return np.array([np.linalg.norm(sommets[:, i] - sommets[:, j]) for i, j in aretes])

for angles in [(0, 0, 0), (25, 35, 0), (123, -48, 77)]:
    L = longueurs_aretes(tourner(cube, *angles), aretes_cube)
    print(f"angles {angles} : longueurs entre {L.min():.12f} et {L.max():.12f}")

### Les curseurs

Dans Jupyter, la cellule suivante affiche des curseurs pour faire tourner le cube et régler la distance de la caméra. Sur le site, ou si `ipywidgets` n'est pas installé, elle affiche une série de vues fixes.

In [ ]:
def afficher(ax_deg=25, ay_deg=35, az_deg=0, d=4.0, f=2.0, sommets=cube, aretes=aretes_cube):
    fig, ax = plt.subplots(figsize=(5, 5))
    p2d, profondeur = projection_perspective(tourner(sommets, ax_deg, ay_deg, az_deg), d=d, f=f)
    tracer(ax, p2d, aretes, f"angles ({ax_deg}, {ay_deg}, {az_deg}), d = {d}", visibles=profondeur > 0.1)
    plt.show()

try:
    from ipywidgets import interact, FloatSlider, IntSlider, fixed
    interact(afficher,
             ax_deg=IntSlider(25, -180, 180, 5, description="rotation x"),
             ay_deg=IntSlider(35, -180, 180, 5, description="rotation y"),
             az_deg=IntSlider(0, -180, 180, 5, description="rotation z"),
             d=FloatSlider(4.0, 1.5, 20.0, 0.5, description="distance"),
             f=FloatSlider(2.0, 0.5, 6.0, 0.5, description="focale"),
             sommets=fixed(cube), aretes=fixed(aretes_cube))
except ImportError:
    print("ipywidgets n'est pas installé : vues fixes à la place des curseurs.")
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    for ax, ay in zip(axes, [0, 30, 60, 90]):
        p2d, prof = projection_perspective(tourner(cube, 20, ay, 0), d=4)
        tracer(ax, p2d, aretes_cube, f"rotation y = {ay}°", visibles=prof > 0.1)
    plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
avant = projection_perspective(np.array([[1.0], [1.0], [-1.0]]), d=4, f=1)[0][0, 0]
arriere = projection_perspective(np.array([[1.0], [1.0], [1.0]]), d=4, f=1)[0][0, 0]
print(f"Votre pari : {pari_rapport_avant_arriere} ; rapport mesuré : {avant / arriere:.3f} (soit 5/3)")

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, d in zip(axes, [1.8, 3, 6, 30]):
    p2d, prof = projection_perspective(tourner(cube, 20, 30), d=d, f=d / 2)
    tracer(ax, p2d, aretes_cube, f"d = {d}, f = d/2", visibles=prof > 0.1)
plt.suptitle("Même taille apparente, caméra de plus en plus loin"); plt.tight_layout(); plt.show()

**Activité** :

1. Expliquez avec la formule $x' = f\,x / (z + d)$ pourquoi la face arrière paraît plus petite. Retrouvez le rapport $5/3$ par le calcul.
2. Sur la série de vues, que devient le dessin quand la caméra s'éloigne (avec une focale qui augmente pour garder la même taille) ? Vers quelle projection tend-il ?
3. En projection orthographique, deux arêtes parallèles restent parallèles. Est-ce encore vrai en perspective ? Que deviennent-elles ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse fragile.** Que se passe-t-il si un sommet passe **derrière** la caméra, c'est-à-dire si $z + d \le 0$ ? Testez `afficher(d=0.8)`. Pourquoi le programme masque-t-il certaines arêtes ? Est-ce une bonne solution ?
- **Vérifier autrement.** La rigidité a été vérifiée sur les longueurs. Vérifiez aussi que chaque matrice de rotation $R$ vérifie $R^{\top} R = I$. Que signifie cette propriété ?
- **Généraliser.** Le moteur ne doit pas dépendre du cube. Créez une **maison** : le cube, plus un toit en pyramide (deux sommets de faîtage, ou une pointe). Écrivez ses sommets et ses arêtes, puis affichez-la avec la même fonction `afficher`. Pour finir, ajoutez des tests : un objet sans arête, une distance négative.

In [ ]:
# Votre maison : sommets (3, n) et liste d'arêtes
maison = None
aretes_maison = None

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
for R in [Rx(37), Ry(-81), Rz(150), tourner(np.eye(3), 10, 20, 30)]:
    assert np.allclose(R.T @ R, np.eye(3))
print("R^T R = I pour toutes les rotations testées : elles conservent longueurs et angles.")

# Maison : le cube, plus deux sommets de faîtage au-dessus de la face y = 1
faitage = np.array([[-1.0, 1.0], [2.0, 2.0], [0.0, 0.0]])
maison = np.hstack([cube, faitage])
indices = {tuple(cube[:, k]): k for k in range(8)}
toit = [(indices[(-1, 1, z)], 8) for z in (-1, 1)] + [(indices[(1, 1, z)], 9) for z in (-1, 1)] + [(8, 9)]
aretes_maison = aretes_cube + toit

def afficher_sur(ax, sommets, aretes, angles=(20, 30, 0), d=6.0, f=3.0):
    if d <= 0:
        raise ValueError("La caméra doit être devant l'objet : d > 0.")
    p2d, prof = projection_perspective(tourner(sommets, *angles), d=d, f=f)
    tracer(ax, p2d, aretes, f"angles {angles}, d = {d}", visibles=prof > 0.1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, angles in zip(axes, [(15, 20, 0), (25, 60, 0), (-10, 140, 0)]):
    afficher_sur(ax, maison * 0.8, aretes_maison, angles)
plt.tight_layout(); plt.show()

try:
    afficher_sur(axes[0], cube, aretes_cube, d=-2)
except ValueError as erreur:
    print("Test distance négative :", erreur)

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Combiner des outils pour produire le résultat": 1,
    "Transformer des constantes en paramètres": 1,
    "Expliquer visuellement": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Quantifier la validité et l'impact des hypothèses": 1,
    "Implémenter un modèle général sous forme de programme robuste": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "M3-projet-moteur-3d", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")